# Host Qwen3.5-9B bằng vLLM trên Kaggle (2× T4) + Cloudflare Tunnel

Server OpenAI-compatible phục vụ RAG pháp luật (ngữ cảnh dài), expose ra internet qua Cloudflare Quick Tunnel.

**Trước khi chạy**
1. `Settings` → **Accelerator: GPU T4 ×2**, **Internet: On**.
2. Chạy ở chế độ **Interactive session** (không dùng *Save & Run All*), vì vLLM và cloudflared là tiến trình nền cần session sống.
3. Nên đặt **Kaggle Secrets** (`Add-ons` → `Secrets`): `HF_TOKEN` (tùy chọn, Qwen3.5-9B là model public) và `VLLM_API_KEY` (khóa bảo vệ endpoint). **Không** ghi token/key thẳng vào notebook.
4. Chạy lần lượt từng cell từ trên xuống. **Mỗi cell khởi chạy chỉ chạy 1 lần.** Nếu cần chạy lại, chạy cell *Dừng toàn bộ* ở cuối trước.

**Ngân sách VRAM (2×T4 ≈ 2×14.5 GiB)**: weights FP16 của 9B ≈ 18 GB, chia đôi bằng tensor-parallel ≈ 9 GB/GPU; phần còn lại dành cho KV cache. Qwen3.5 là kiến trúc hybrid (Gated DeltaNet + full attention xen kẽ) nên KV cache theo độ dài ngữ cảnh khá nhẹ, phù hợp văn bản pháp luật dài.

## 1. Cài vLLM

In [1]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("vllm") is None:
    # Cảnh báo "dependency conflicts" (protobuf, cuda-python, RAPIDS...) của image Kaggle là bình thường, không ảnh hưởng vLLM.
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
    print("pip exit code:", r.returncode)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
else:
    print("vLLM đã có sẵn, bỏ qua cài đặt.")

import vllm  # chỉ import vllm (không import torch) để tiến trình notebook không chiếm CUDA context
print("vLLM version:", vllm.__version__)

pip exit code: 0
vLLM version: 0.30.0


## 2. Cấu hình
Chỉnh các tham số trong `CFG`. Giá trị mặc định đã tính cho Qwen3.5-9B trên 2×T4.

In [2]:
import os, sys, json, time, re, shutil, signal, secrets, socket, subprocess
from pathlib import Path
import requests

# ---------- Secrets (KHÔNG hardcode) ----------
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, "")

HF_TOKEN = get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

API_KEY = get_secret("VLLM_API_KEY")
if API_KEY:
    print("API key: lấy từ Kaggle Secret VLLM_API_KEY")
else:
    API_KEY = secrets.token_urlsafe(32)
    print("Chưa có Secret VLLM_API_KEY -> tạo key ngẫu nhiên cho session này (đổi mỗi lần chạy):")
    print("   ", API_KEY)

# ---------- Cấu hình server ----------
CFG = dict(
    model="Qwen/Qwen3.5-9B",
    host="127.0.0.1",          # chỉ cần localhost vì cloudflared chạy cùng máy
    port=8000,
    dtype="half",              # T4 (sm_75) không hỗ trợ bf16 -> dùng fp16
    gpu_memory_utilization=0.90,
    max_model_len=16384,       # ngữ cảnh tối đa (prompt + output). Nếu OOM/KV không đủ: 24576 hoặc 16384
    max_num_seqs=16,           # số request đồng thời tối đa; ngữ cảnh dài thì không nên đặt cao
    max_num_batched_tokens=4096,  # chunk prefill nhỏ -> đỉnh VRAM activation thấp hơn
    enable_thinking=False,     # RAG thường không cần chế độ suy luận dài; client vẫn bật lại được theo từng request
    enable_prefix_caching=False,  # Qwen3.5 hybrid cần thêm bộ nhớ khi bật prefix cache; bật lại nếu còn dư VRAM
)

LOG_FILE = Path("/kaggle/working/vllm_server.log")
BASE_URL = f"http://127.0.0.1:{CFG['port']}"
AUTH = {"Authorization": f"Bearer {API_KEY}"}

# ---------- Đếm GPU bằng nvidia-smi (không khởi tạo CUDA trong notebook) ----------
def gpu_info():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True).stdout.strip().splitlines()
    return [(n.strip(), float(t), float(f)) for n, t, f in (l.split(",") for l in out)]

GPUS = gpu_info()
NUM_GPUS = len(GPUS)
for i, (n, t, f) in enumerate(GPUS):
    print(f"GPU {i}: {n} | total {t/1024:.1f} GiB | free {f/1024:.1f} GiB")
if NUM_GPUS < 2:
    raise RuntimeError("Cần 2 GPU: Settings -> Accelerator -> GPU T4 x2. Model 9B FP16 không vừa 1 GPU T4.")
print(f"\nSẽ host {CFG['model']} với tensor-parallel-size={NUM_GPUS}")

API key: lấy từ Kaggle Secret VLLM_API_KEY
GPU 0: Tesla T4 | total 15.0 GiB | free 14.6 GiB
GPU 1: Tesla T4 | total 15.0 GiB | free 14.6 GiB

Sẽ host Qwen/Qwen3.5-9B với tensor-parallel-size=2


## 3. Kiểm tra trước khi chạy (dọn tiến trình cũ, kiểm tra VRAM và port)

In [3]:
# Dọn tiến trình vLLM/cloudflared còn sót từ lần chạy trước (nguyên nhân phổ biến gây lỗi 'Free memory ... is less than desired')
for pattern in ("vllm serve", "vllm.entrypoints", "VLLM::", "cloudflared tunnel"):
    subprocess.run(["pkill", "-9", "-f", pattern], capture_output=True)
time.sleep(5)

def port_in_use(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if port_in_use(CFG["port"]):
    raise RuntimeError(f"Port {CFG['port']} đang bị chiếm. Restart session rồi chạy lại.")

need_gib = CFG["gpu_memory_utilization"] * GPUS[0][1] / 1024
for i, (n, t, f) in enumerate(gpu_info()):
    print(f"GPU {i}: free {f/1024:.2f} GiB (vLLM cần ≥ {need_gib:.2f} GiB)")
    if f / 1024 < need_gib:
        raise RuntimeError(
            f"GPU {i} còn quá ít VRAM trống. Có tiến trình khác đang giữ GPU -> "
            "dùng Run -> Restart session (hoặc Factory reset) rồi chạy lại từ đầu.")
print("OK, sẵn sàng khởi chạy.")

GPU 0: free 14.56 GiB (vLLM cần ≥ 13.50 GiB)
GPU 1: free 14.56 GiB (vLLM cần ≥ 13.50 GiB)
OK, sẵn sàng khởi chạy.


## 4. Khởi chạy vLLM server (chỉ chạy cell này **một lần**)

In [4]:
if "vllm_proc" in globals() and vllm_proc.poll() is None:
    raise RuntimeError(f"vLLM đang chạy (PID={vllm_proc.pid}). Không khởi chạy lần 2 — chạy cell 'Dừng toàn bộ' nếu muốn khởi động lại.")

vllm_bin = shutil.which("vllm")
if vllm_bin is None:
    raise RuntimeError("Không tìm thấy lệnh 'vllm'. Chạy lại cell cài đặt.")

cmd = [
    vllm_bin, "serve", CFG["model"],
    "--served-model-name", CFG["model"],
    "--host", CFG["host"],
    "--port", str(CFG["port"]),
    "--tensor-parallel-size", str(NUM_GPUS),
    "--dtype", CFG["dtype"],
    "--gpu-memory-utilization", str(CFG["gpu_memory_utilization"]),
    "--max-model-len", str(CFG["max_model_len"]),
    "--max-num-seqs", str(CFG["max_num_seqs"]),
    "--max-num-batched-tokens", str(CFG["max_num_batched_tokens"]),
    "--language-model-only",      # bỏ vision encoder: tiết kiệm VRAM + bỏ ~45s warmup đa phương thức
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", json.dumps({"enable_thinking": CFG["enable_thinking"]}),
    "--enforce-eager",            # bỏ CUDA graph: tiết kiệm VRAM, ổn định hơn trên T4
    "--enable-prefix-caching" if CFG["enable_prefix_caching"] else "--no-enable-prefix-caching",
]

env = os.environ.copy()
env["VLLM_API_KEY"] = API_KEY                      # truyền key qua env, không lộ trong `ps aux`
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# Nếu treo ở bước khởi tạo NCCL, thử bỏ comment dòng dưới:
# env["NCCL_P2P_DISABLE"] = "1"

print("Lệnh:", " ".join(cmd))
log_f = open(LOG_FILE, "w")
vllm_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env, start_new_session=True)
print(f"\nĐã khởi chạy vLLM (PID={vllm_proc.pid}). Log: {LOG_FILE}")

Lệnh: /usr/local/bin/vllm serve Qwen/Qwen3.5-9B --served-model-name Qwen/Qwen3.5-9B --host 127.0.0.1 --port 8000 --tensor-parallel-size 2 --dtype half --gpu-memory-utilization 0.9 --max-model-len 16384 --max-num-seqs 16 --max-num-batched-tokens 4096 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs {"enable_thinking": false} --enforce-eager --no-enable-prefix-caching

Đã khởi chạy vLLM (PID=210). Log: /kaggle/working/vllm_server.log


## 5. Chờ server sẵn sàng
Lần đầu phải tải ~19 GB weights từ Hugging Face nên có thể mất 5–15 phút.

In [5]:
def tail(path, n=60):
    try:
        return "".join(open(path, errors="replace").readlines()[-n:])
    except FileNotFoundError:
        return ""

def wait_ready(proc, timeout=1800):
    t0, last_print = time.time(), 0
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            print(f"\n❌ vLLM đã thoát với exit code {proc.returncode}. 60 dòng log cuối:\n")
            print(tail(LOG_FILE, 60))
            raise RuntimeError("vLLM crash — xem log ở trên (gợi ý ở cuối notebook).")
        try:
            if requests.get(f"{BASE_URL}/health", timeout=3).status_code == 200:
                print(f"\n✅ vLLM sẵn sàng sau {int(time.time()-t0)}s")
                return
        except requests.RequestException:
            pass
        if time.time() - last_print >= 15:
            last = (tail(LOG_FILE, 1).strip() or "...")[-110:]
            print(f"⏳ {int(time.time()-t0):>4}s | {last}")
            last_print = time.time()
        time.sleep(3)
    raise TimeoutError("Hết thời gian chờ vLLM khởi động.")

wait_ready(vllm_proc)

⏳    0s | ...
⏳   15s | ...
⏳   30s | ...
⏳   45s | nable_prefix_caching': False, 'language_model_only': True, 'max_num_batched_tokens': 4096, 'max_num_seqs': 16}
⏳   60s | (APIServer pid=210) INFO 10-03 16:27:12 [compilation.py:336] Enabled custom fusions: norm_quant, act_quant
⏳   75s | (APIServer pid=210) INFO 10-03 16:27:12 [compilation.py:336] Enabled custom fusions: norm_quant, act_quant
⏳   90s | de_rank_within_dp=0, master_addr=127.0.0.1, mq_connect_ip=172.19.2.2 (local), world_size=2, local_world_size=2
⏳  105s | ank=0 local_rank=0 distributed_init_method=file:///tmp/vllm_dist_4cda850d0e1a4d72872230457eef0fab backend=nccl
⏳  120s | cuda.py:538] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
⏳  135s | cuda.py:538] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
⏳  150s | cuda.py:538] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION']

In [6]:
# Dung lượng KV cache thực tế mà vLLM cấp được
for line in LOG_FILE.read_text(errors="replace").splitlines():
    if any(k in line for k in ("KV cache size", "Maximum concurrency", "Available KV cache memory", "Model loading took")):
        print(re.sub(r"^\(.*?\)\s*", "", line))

INFO 10-03 16:30:44 [model_runner.py:428] Model loading took 8.46 GiB memory and 163.626895 seconds
INFO 10-03 16:30:44 [model_runner.py:428] Model loading took 8.46 GiB memory and 163.643862 seconds
INFO 10-03 16:37:23 [gpu_worker.py:640] Available KV cache memory: 3.68 GiB
INFO 10-03 16:37:23 [kv_cache_utils.py:2395] GPU KV cache size: 213,460 tokens, Maximum concurrency for 16,384 tokens per request: 13.03x


## 6. Kiểm tra server cục bộ

In [7]:
import os
import psutil
import requests
try:
    import torch
    HAS_TORCH = torch.cuda.is_available()
except ImportError:
    HAS_TORCH = False

def get_memory_info():
    """Hàm bổ trợ lấy thông tin bộ nhớ RAM (Client/Process) và GPU (nếu có PyTorch/CUDA)"""
    # RAM hệ thống
    process = psutil.Process(os.getpid())
    ram_mb = process.memory_info().rss / (1024 * 1024)
    sys_ram = psutil.virtual_memory()
    info = f"RAM Client Process: {ram_mb:.1f} MB | Hệ thống đã dùng: {sys_ram.percent}%"
    
    # GPU Memory (Nếu script chạy trực tiếp trên máy host GPU)
    if HAS_TORCH:
        gpu_allocated = torch.cuda.memory_allocated() / (1024 ** 2)
        gpu_reserved = torch.cuda.memory_reserved() / (1024 ** 2)
        info += f" || GPU Allocated: {gpu_allocated:.1f} MB | Reserved: {gpu_reserved:.1f} MB"
        
    return info


In [8]:
# Gửi request Chat Completions
resp = requests.post(
    f"{BASE_URL}/v1/chat/completions",
    headers=AUTH,
    timeout=300,
    json={
        "model": CFG["model"],
        "messages": [
            {"role": "system", "content": "Bạn là chuyên gia về pháp luật Việt Nam. Hãy trả lời đúng trọng tâm câu hỏi từ văn bản đã có và cho biết nó thuộc điều nào khoản nào điêm nào ."},
            {"role": "user", "content": """
            Câu hỏi: Đi xe máy vượt đèn đỏ phạt bao nhiêu tiền trong giao thông đường bộ ?
 Văm bản tìm thấy           
Số ký hiệu : 168/2024/NĐ-CP
Tiêu đề    : Nghị định số 168/2024/NĐ-CP Quy định xử phạt vi phạm hành chính về trật tự, an toàn giao thông trong lĩnh vực giao thông đường bộ; trừ điểm, phục hồi điểm giấy phép lái xe
Trạng thái : Còn hiệu lực
Nội dung  : Điều 9. Xử phạt người điều khiển xe đạp, xe đạp máy, người điều khiển xe thô sơ khác vi phạm quy tắc giao thông đường bộ
1. Phạt tiền từ 100.000 đồng đến 200.000 đồng đối với người điều khiển xe thực hiện một trong các hành vi vi phạm sau đây:
a) Không đi bên phải theo chiều đi của mình, đi không đúng phần đường quy định;
b) Dừng xe đột ngột; chuyển hướng không báo hiệu trước;
c) Không chấp hành hiệu lệnh hoặc chỉ dẫn của biển báo hiệu, vạch kẻ đường, trừ các hành vi vi phạm quy định tại điểm đ khoản 2, điểm c khoản 3 Điều này;
d) Vượt bên phải trong các trường hợp không được phép;
đ) Dừng xe, đỗ xe trên phần đường xe chạy ở đoạn đường ngoài đô thị nơi có lề đường;
e) Chạy trong hầm đường bộ không bật đèn hoặc không có vật phát sáng báo hiệu; dừng xe, đỗ xe trong hầm đường bộ không đúng nơi quy định; quay đầu xe trong hầm đường bộ;
g) Điều khiển xe đạp, xe đạp máy đi dàn hàng ngang từ 03 xe trở lên, xe thô sơ khác đi dàn hàng ngang từ 02 xe trở lên;
h) Người điều khiển xe đạp, xe đạp máy sử dụng ô (dù), dùng tay cầm và sử dụng điện thoại hoặc các thiết bị điện tử khác; chở người ngồi trên xe đạp, xe đạp máy sử dụng ô (dù);
i) Điều khiển xe thô sơ trong thời gian từ 18 giờ ngày hôm trước đến 06 giờ ngày hôm sau không sử dụng đèn hoặc không có báo hiệu ở phía trước và phía sau xe;
k) Để xe ở lòng đường, vỉa hè trái phép; đỗ xe ở lòng đường gây cản trở giao thông, đỗ xe trên cầu gây cản trở giao thông;
l) Không tuân thủ các quy định về dừng xe, đỗ xe tại nơi đường bộ giao nhau cùng mức với đường sắt;
m) Dùng xe đẩy làm quầy hàng lưu động trên đường gây cản trở giao thông;
n) Không giảm tốc độ (hoặc dừng lại) và nhường đường khi điều khiển xe đi từ đường không ưu tiên ra đường ưu tiên, từ đường nhánh ra đường chính;
o) Xe đạp, xe đạp máy, xe xích lô chở quá số người quy định, trừ trường hợp chở người bệnh đi cấp cứu;
p) Điều khiển xe trên đường mà trong máu hoặc hơi thở có nồng độ cồn nhưng chưa vượt quá 50 miligam/100 mililít máu hoặc chưa vượt quá 0,25 miligam/1 lít khí thở.
2. Phạt tiền từ 150.000 đồng đến 250.000 đồng đối với người điều khiển xe thực hiện một trong các hành vi vi phạm sau đây:
a) Điều khiển xe đạp, xe đạp máy buông cả hai tay; chuyển hướng đột ngột trước đầu xe cơ giới đang chạy; dùng chân điều khiển xe đạp, xe đạp máy;
b) Không chấp hành hiệu lệnh, chỉ dẫn của người điều khiển giao thông hoặc người kiểm soát giao thông;
c) Người đang điều khiển xe hoặc chở người ngồi trên xe bám, kéo, đẩy xe khác, vật khác, mang vác vật cồng kềnh; điều khiển xe kéo theo xe khác, vật khác;
d) Không nhường đường cho xe xin vượt khi có đủ điều kiện an toàn hoặc gây cản trở đối với xe cơ giới xin vượt, gây cản trở xe ưu tiên;
đ) Không chấp hành hiệu lệnh của đèn tín hiệu giao thông.
3. Phạt tiền từ 300.000 đồng đến 400.000 đồng đối với người điều khiển xe thực hiện một trong các hành vi vi phạm sau đây:
a) Điều khiển xe lạng lách, đánh võng; đuổi nhau trên đường;
b) Đi xe bằng một bánh đối với xe đạp, xe đạp máy; đi xe bằng hai bánh đối với xe xích lô;
c) Đi vào khu vực cấm, đường có biển báo hiệu nội dung cấm đi vào đối với loại phương tiện đang điều khiển; đi ngược chiều đường của đường một chiều, đường có biển "Cấm đi ngược chiều";
d) Điều khiển xe trên đường mà trong máu hoặc hơi thở có nồng độ cồn vượt quá 50 miligam đến 80 miligam/100 mililít máu hoặc vượt quá 0,25 miligam đến 0,4 miligam/1 lít khí thở.
4. Phạt tiền từ 400.000 đồng đến 600.000 đồng đối với người điều khiển xe thực hiện một trong các hành vi vi phạm sau đây:
a) Gây tai nạn giao thông không dừng ngay phương tiện, không giữ nguyên hiện trường, không trợ giúp người bị nạn, không ở lại hiện trường hoặc không đến trình báo ngay với cơ quan công an, Ủy ban nhân dân nơi gần nhất;
b) Điều khiển xe trên đường mà trong máu hoặc hơi thở có nồng độ cồn vượt quá 80 miligam/100 mililít máu hoặc vượt quá 0,4 miligam/1 lít khí thở;
c) Không chấp hành yêu cầu kiểm tra về nồng độ cồn của người thi hành công vụ;
d) Người điều khiển xe đạp máy không đội "mũ bảo hiểm cho người đi mô tô, xe máy" hoặc đội "mũ bảo hiểm cho người đi mô tô, xe máy" không cài quai đúng quy cách khi tham gia giao thông trên đường bộ;
đ) Chở người ngồi trên xe đạp máy không đội "mũ bảo hiểm cho người đi mô tô, xe máy" hoặc đội "mũ bảo hiểm cho người đi mô tô, xe máy" không cài quai đúng quy cách, trừ trường hợp chở người bệnh đi cấp cứu, trẻ em dưới 06 tuổi, áp giải người có hành vi vi phạm pháp luật.
5. Phạt tiền từ 800.000 đồng đến 1.200.000 đồng đối với người điều khiển xe đi vào đường cao tốc, trừ phương tiện phục vụ việc quản lý, bảo trì đường cao tốc.
            
            """},
        ],
        "max_tokens": 300,
        "temperature": 0.2,
    }
).json()

msg = resp["choices"][0]["message"]
print("\n--- NỘI DUNG MÔ HÌNH TRẢ LỜI ---")
print(msg["content"])

# 3. In số tokens và usage chi tiết
usage = resp.get("usage", {})
prompt_tokens = usage.get("prompt_tokens", 0)
completion_tokens = usage.get("completion_tokens", 0)
total_tokens = usage.get("total_tokens", 0)

print("\n--- THÔNG TIN TOKENS & BỘ NHỚ ---")
print(f"Prompt tokens: {prompt_tokens}")
print(f"Completion tokens: {completion_tokens}")
print(f"Total tokens: {total_tokens}")
print(f"Usage chi tiết: {usage}")
print(f"Bộ nhớ sử dụng: {get_memory_info()}")

# 4. Kiểm tra lỗi output
txt = msg["content"] or ""
if len(txt.strip()) < 5 or len(set(txt)) < 8:
    print("\n⚠️ Output đáng ngờ (rỗng/lặp ký tự). Có thể fp16 bị tràn số — xem mục Xử lý sự cố ở cuối.")


--- NỘI DUNG MÔ HÌNH TRẢ LỜI ---
Chào bạn, dựa trên văn bản **Nghị định số 168/2024/NĐ-CP** mà bạn cung cấp, tôi xin trả lời câu hỏi của bạn như sau:

**1. Mức phạt tiền:**
Việc đi xe máy vượt đèn đỏ bị phạt tiền từ **1.000.000 đồng đến 2.000.000 đồng**.

*Lưu ý quan trọng:* Trong nội dung văn bản bạn cung cấp (Điều 9), điểm **đ** khoản 2 quy định hành vi "Không chấp hành hiệu lệnh của đèn tín hiệu giao thông" nhưng chỉ áp dụng cho **người điều khiển xe đạp, xe đạp máy** (phạt 150.000 - 250.000 đồng). Tuy nhiên, đối với **xe máy** (thuộc nhóm xe cơ giới), hành vi này được quy định tại **Điều 10** của cùng Nghị định này (không có trong đoạn văn bản bạn trích dẫn). Theo quy định chung của pháp luật hiện hành về xe máy, mức phạt là 1.000.000 - 2.000.000 đồng.

**2. Vị trí pháp lý cụ thể:**
Hành vi này thuộc:
*   **Văn bản:** Nghị định số 168/2024/NĐ-CP.
*  

--- THÔNG TIN TOKENS & BỘ NHỚ ---
Prompt tokens: 1512
Completion tokens: 300
Total tokens: 1812
Usage chi tiết: {'prompt_tokens': 1

## 7. Cloudflare Tunnel
Chỉ khởi tạo **sau khi** vLLM đã sẵn sàng.

In [9]:
CF_BIN, CF_LOG = Path("/tmp/cloudflared"), Path("/tmp/cloudflared.log")

if not CF_BIN.exists():
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(CF_BIN, "wb") as f:
            shutil.copyfileobj(r.raw, f)
    CF_BIN.chmod(0o755)
print(subprocess.run([str(CF_BIN), "--version"], capture_output=True, text=True).stdout.strip())

if "cf_proc" in globals() and cf_proc.poll() is None:
    raise RuntimeError("cloudflared đang chạy rồi — dùng PUBLIC_URL hiện có hoặc chạy cell 'Dừng toàn bộ'.")

cf_log = open(CF_LOG, "w")
cf_proc = subprocess.Popen(
    [str(CF_BIN), "tunnel", "--url", BASE_URL, "--no-autoupdate"],
    stdout=cf_log, stderr=subprocess.STDOUT, start_new_session=True)

PUBLIC_URL, pat = None, re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")
for _ in range(60):
    if cf_proc.poll() is not None:
        print(tail(CF_LOG, 30)); raise RuntimeError("cloudflared thoát bất thường.")
    m = pat.search(CF_LOG.read_text(errors="replace"))
    if m:
        PUBLIC_URL = m.group(0); break
    time.sleep(1)

if not PUBLIC_URL:
    print(tail(CF_LOG, 30)); raise RuntimeError("Không lấy được public URL sau 60s.")
print("Public URL:", PUBLIC_URL)

cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)
Public URL: https://five-occur-consent-batman.trycloudflare.com


In [10]:
# Kiểm tra qua tunnel (URL mới có thể cần vài giây để DNS lan truyền)
for attempt in range(20):
    try:
        r = requests.get(f"{PUBLIC_URL}/v1/models", headers=AUTH, timeout=10)
        if r.status_code == 200:
            print("✅ Tunnel hoạt động:", [m["id"] for m in r.json()["data"]])
            break
        print("HTTP", r.status_code)
    except requests.RequestException as e:
        pass
    time.sleep(3)
else:
    print("⚠️ Chưa gọi được qua tunnel. Xem log:\n", tail(CF_LOG, 20))

shown_key = API_KEY if not get_secret("VLLM_API_KEY") else "<VLLM_API_KEY của bạn>"
print(f'''
===== Dùng từ máy local =====
from openai import OpenAI
client = OpenAI(base_url="{PUBLIC_URL}/v1", api_key="{shown_key}", timeout=600)

stream = client.chat.completions.create(
    model="{CFG["model"]}",
    messages=[{{"role": "user", "content": "Xin chào!"}}],
    max_tokens=1024,
    stream=True,   # nên stream: Cloudflare ngắt request không có dữ liệu trả về sau ~100s
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")
''')

HTTP 530
✅ Tunnel hoạt động: ['Qwen/Qwen3.5-9B']

===== Dùng từ máy local =====
from openai import OpenAI
client = OpenAI(base_url="https://five-occur-consent-batman.trycloudflare.com/v1", api_key="<VLLM_API_KEY của bạn>", timeout=600)

stream = client.chat.completions.create(
    model="Qwen/Qwen3.5-9B",
    messages=[{"role": "user", "content": "Xin chào!"}],
    max_tokens=1024,
    stream=True,   # nên stream: Cloudflare ngắt request không có dữ liệu trả về sau ~100s
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")



## 8. Ghi chú sử dụng

**Bật chế độ suy luận (thinking) theo từng request** (mặc định đang tắt cho RAG):
```python
client.chat.completions.create(
    model="Qwen/Qwen3.5-9B", messages=[...],
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)
# phần suy luận nằm ở message.reasoning_content / delta.reasoning_content
```

**Tham số sinh gợi ý cho RAG pháp luật** (không bật thinking): `temperature` 0–0.3, `max_tokens` đủ cho câu trả lời có trích dẫn điều/khoản. Prompt nên yêu cầu "chỉ trả lời dựa trên ngữ cảnh, trích dẫn điều khoản, nếu không có thì nói không có".

**Giới hạn cần biết**
- URL `trycloudflare.com` đổi mỗi lần chạy lại cell tunnel; chết khi session Kaggle kết thúc (tối đa vài giờ).
- Quick Tunnel không có SLA, không phù hợp production. Dùng cho dev/thử nghiệm.
- Cloudflare ngắt kết nối nếu ~100s không có byte nào trả về → luôn dùng `stream=True` với prompt dài.

**Xử lý sự cố**
| Triệu chứng | Cách xử lý |
|---|---|
| `Free memory ... is less than desired` | Còn tiến trình cũ giữ GPU → Restart session. Hoặc hạ `gpu_memory_utilization` xuống 0.85 |
| `CUDA out of memory` khi load/warmup | Hạ `max_model_len` (24576 → 16384), `max_num_batched_tokens` (2048), `max_num_seqs` (8) |
| `larger than the maximum number of tokens that can be stored in KV cache` | Log sẽ in `estimated maximum model length` → đặt `max_model_len` ≤ giá trị đó |
| Output rỗng / ký tự rác | fp16 tràn số trên T4. Thử model đã lượng tử hóa (AWQ/GPTQ 4-bit của Qwen3.5-9B) — khi đó có thể chạy 1 GPU, `tensor-parallel-size=1` |
| Treo ở khởi tạo NCCL | Bật `NCCL_P2P_DISABLE=1` (dòng comment trong cell khởi chạy) |